In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input,LSTM,Embedding,Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
import contractions

import warnings as wr
wr.filterwarnings('ignore')

#from tensorflow.keras import mixed_precision
#mixed_precision.set_global_policy('mixed_float16')

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [3]:
df = pd.read_csv("Dataset_English_Hindi.csv")

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
#lower case
df['English'] = df['English'].str.lower()

In [6]:
#remove non english words
def remove_non_english(text):
    pattern = r'[^a-zA-Z0-9\s]'
    text = re.sub(pattern, '', text)
    return text

df['English'] = df['English'].astype(str)
df['English'] = df['English'].apply(remove_non_english)

In [7]:
#remove non hindi words
def remove_non_hindi(text):
    pattern = r'[^\u0900-\u097F\s]'
    text = re.sub(pattern, '', text)
    return text

df['Hindi'] = df['Hindi'].astype(str)
df['Hindi'] = df['Hindi'].apply(remove_non_hindi)

In [8]:
#remove urls
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'',text)

df['English'] = df['English'].apply(remove_url)
df['Hindi'] = df['Hindi'].apply(remove_url)

In [9]:
#handle chat words
chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "KISS": "Keep It Simple, Stupid",
    "LDR": "Long Distance Relationship",
    "LMAO": "Laugh My A.. Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laughter",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "LOL": "Laughing out loud",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don’t care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "IDC": "I don’t care",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "TIME": "Tears in my eyes",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "LMAO": "Laughing my a** off",
    "BFF": "Best friends forever",
    "CSL": "Can’t stop laughing",
}

In [10]:
def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words.keys():
            new_text.append(chat_words[w.upper()].lower())
        else:
            new_text.append(w)
    return ' '.join(new_text)

df['English'] = df['English'].apply(chat_conversion)

In [11]:
#remove emojis
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

df['English'] = df['English'].apply(remove_emoji)
df['Hindi'] = df['Hindi'].apply(remove_emoji)

In [12]:
#expand contractions
def expand_contractions(text):
    expanded_text = contractions.fix(text)
    return expanded_text

df['English'] = df['English'].apply(expand_contractions)

In [13]:
#tokenizer english
tok_eng = Tokenizer(filters='', oov_token='<unk>')
tok_eng.fit_on_texts(df['English'])
X = tok_eng.texts_to_sequences(df['English'])
X = pad_sequences(X)

In [14]:
len(tok_eng.word_index)

74272

In [15]:
X.shape

(127688, 401)

In [16]:
latent_dim = 256

In [17]:
#tokenizer hindi
df['Hindi'] = '<sos> ' + df['Hindi'] + ' <eos>'
tok_hin = Tokenizer(filters='', oov_token='<unk>')
tok_hin.fit_on_texts(df['Hindi'])
y = tok_hin.texts_to_sequences(df['Hindi'])
y = pad_sequences(y)

In [18]:
len(tok_hin.word_index)

77014

In [19]:
y.shape

(127688, 418)

In [20]:
vocab_eng = len(tok_eng.word_index) + 1
vocab_hin = len(tok_hin.word_index) + 1

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [22]:
# Shifted decoder input/target
decoder_input_train  = y_train[:, :-1]
decoder_target_train = y_train[:, 1:]
decoder_input_test   = y_test[:, :-1]
decoder_target_test  = y_test[:, 1:]

max_encoder_len = X_train.shape[1]
max_decoder_len = decoder_input_train.shape[1]

In [23]:
# --- Encoder ---
enc_inputs = Input(shape=(max_encoder_len,), name="enc_input")
enc_embed  = Embedding(input_dim=vocab_eng, output_dim=latent_dim, name="enc_embed")(enc_inputs)
enc_outputs, state_h, state_c = LSTM(latent_dim, return_state=True, name="enc_lstm")(enc_embed)
enc_states = [state_h, state_c]

In [24]:
# --- Decoder ---
dec_inputs = Input(shape=(max_decoder_len,), name="dec_input")
dec_embed  = Embedding(input_dim=vocab_hin, output_dim=latent_dim, name="dec_embed")(dec_inputs)
dec_lstm   = LSTM(latent_dim, return_sequences=True, return_state=True, name="dec_lstm")
dec_outputs, _, _ = dec_lstm(dec_embed, initial_state=enc_states)
dec_dense  = Dense(vocab_hin, activation='softmax', name="dec_dense")
dec_outputs = dec_dense(dec_outputs)

In [25]:
#model
model = Model([enc_inputs, dec_inputs], dec_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 enc_input (InputLayer)         [(None, 401)]        0           []                               
                                                                                                  
 dec_input (InputLayer)         [(None, 417)]        0           []                               
                                                                                                  
 enc_embed (Embedding)          (None, 401, 256)     19013888    ['enc_input[0][0]']              
                                                                                                  
 dec_embed (Embedding)          (None, 417, 256)     19715840    ['dec_input[0][0]']              
                                                                                              

#### Training

In [ ]:
#training
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

with tf.device('/GPU:0'):
    history = model.fit(
        x=[X_train, decoder_input_train],
        y=decoder_target_train,
        batch_size=64,
        epochs=1,
        validation_data=([X_test, decoder_input_test], decoder_target_test),
        callbacks=[early_stopping]
    )

#### Saving Model

In [ ]:
model.save("my_model.keras")
model.save_weights("my_model.weights.h5")
with open("my_model_architecture.json", 'w') as f:
    f.write(model.to_json())


import pickle
# Save
with open('tok_eng.pickle', 'wb') as handle:
    pickle.dump(tok_eng, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open('tok_hin.pickle', 'wb') as handle:
    pickle.dump(tok_hin, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Loading Saved Model

model = load_model('my_model.keras')

## Translation

In [27]:
encoder_model = Model(enc_inputs, enc_states)

dec_state_h = Input(shape=(latent_dim,), name="dec_h")
dec_state_c = Input(shape=(latent_dim,), name="dec_c")
dec_states = [dec_state_h, dec_state_c]

dec_inf_input = Input(shape=(1,), name="dec_inf")
dec_inf_embed = model.get_layer("dec_embed")(dec_inf_input)
dec_inf_lstm_out, dec_inf_h, dec_inf_c = model.get_layer("dec_lstm")(
    dec_inf_embed, initial_state=dec_states
)
dec_inf_dense = model.get_layer("dec_dense")(dec_inf_lstm_out)

decoder_model = Model([dec_inf_input] + dec_states, 
                      [dec_inf_dense, dec_inf_h, dec_inf_c])

In [28]:
hin_index_to_word = {v: k for k, v in tok_hin.word_index.items()}
sos_token = tok_hin.word_index['<sos>']
eos_token = tok_hin.word_index['<eos>']

def preprocess_english(s):
    s = s.lower()
    s = remove_non_english(s)  # your function
    s = remove_url(s)
    s = chat_conversion(s)
    s = remove_emoji(s)
    s = expand_contractions(s)
    seq = tok_eng.texts_to_sequences([s])
    return pad_sequences(seq, maxlen=max_encoder_len)

In [29]:
def translate_sentence(eng_sentence):
    input_seq = preprocess_english(eng_sentence)
    states = encoder_model.predict(input_seq, verbose=0)
    
    target_seq = np.array([[sos_token]])
    decoded = []
    
    for _ in range(max_decoder_len):
        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        token_idx = np.argmax(output[0, 0, :])
        
        if token_idx == eos_token:
            break
            
        word = hin_index_to_word.get(token_idx, '')
        if word not in ('<unk>', ''):
            decoded.append(word)
            
        target_seq = np.array([[token_idx]])
        states = [h, c]
    
    return ' '.join(decoded)

In [30]:
def translate_batch(sentences):
    return [translate_sentence(s) for s in sentences]

In [31]:
# Example
sentences = [
    "How are you?",
    "I love cricket.",
    "The weather is nice today.",
    "The quick brown fox jumped over a lazy dog.",
    "It's always sunny in Philadelphia.",
    "The only thing that matters in this world is happiness.",
    "What is the syllabus of machine learning this year.",
    "Once upon a time, there was a king who was very powerful.",
    "Are there any rats in this house?",
    "This task is a piece of cake.",
    "The wife of my brother is winning trophies in sports.",
    "I am very happy today, I have become a father.",
    "There are four members in my family, two live in Delhi and the other two live in Australia."
]
hindi_translations = translate_batch(sentences)
for en, hi in zip(sentences, hindi_translations):
    print("EN:", en)
    print("HI:", hi)
    print()

TypeError: in user code:

    File "C:\Users\Grv\anaconda3\envs\tf_gpu\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "C:\Users\Grv\anaconda3\envs\tf_gpu\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "C:\Users\Grv\anaconda3\envs\tf_gpu\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "C:\Users\Grv\anaconda3\envs\tf_gpu\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "C:\Users\Grv\anaconda3\envs\tf_gpu\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "C:\Users\Grv\anaconda3\envs\tf_gpu\lib\site-packages\keras\backend.py", line 2455, in dot
        out = tf.ma